In [1]:
!pip -q uninstall -y torchao
!pip -q install peft

In [2]:
import os, re, json, time, zipfile, random
from bisect import bisect_right
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

PHASE1_REPO = "Granth2/cs781-phase1-Submission-9"
DATA_REPO = "Exploration-Lab/CS-781-Capstone"

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    raise Exception("HF Token could not be found in Kaggle Secrets!")

N_EMBD, N_LAYER, N_HEAD, Context_window = 768, 12, 12, 1024

MAX_LEN, STRIDE = 512, 128
KEEP_EMPTY_PROB = 0.3
EPOCHS, LR, BATCH = 10, 2e-4, 8

device: cuda


In [3]:
%%writefile attention.py
import math
from typing import Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.1, attention_scale: float = None,
                 use_rope: bool = True, rope_base: float = 10000.0):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be evenly divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.gate_proj = nn.Linear(d_model, d_model, bias=False)

        self.use_rope = use_rope
        if use_rope:
            assert self.d_head % 2 == 0, "RoPE needs an even head dimension"
            inv_freq = 1.0 / (rope_base ** (torch.arange(0, self.d_head, 2, dtype=torch.float32) / self.d_head))
            pos_steps = torch.arange(max_seq_len, dtype=torch.float32)
            angles = torch.outer(pos_steps, inv_freq)
            self.register_buffer("rope_cos", torch.cos(angles), persistent=False)
            self.register_buffer("rope_sin", torch.sin(angles), persistent=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
            persistent=False
        )

    def _apply_rope(self, x: torch.Tensor, start: int) -> torch.Tensor:
        seq_len = x.size(2)
        cos = self.rope_cos[start:start + seq_len].float()
        sin = self.rope_sin[start:start + seq_len].float()
        half_dim = self.d_head // 2
        x1, x2 = x[..., :half_dim].float(), x[..., half_dim:].float()
        rotated = torch.cat((x1 * cos - x2 * sin, x1 * sin + x2 * cos), dim=-1)
        return rotated.to(x.dtype)

    def forward(
        self,
        x: torch.Tensor,
        past_kv: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
        use_cache: bool = False,
        key_padding_mask: Optional[torch.Tensor] = None,
    ):
        bsz, seq_len, dim = x.shape

        q = self.q_proj(x).view(bsz, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(bsz, seq_len, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(bsz, seq_len, self.n_heads, self.d_head).transpose(1, 2)

        offset = 0 if past_kv is None else past_kv[0].size(2)

        if self.use_rope:
            q = self._apply_rope(q, offset)
            k = self._apply_rope(k, offset)

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        cached_kv = (k, v) if use_cache else None

        if key_padding_mask is None and offset == 0:
            out = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=True,
                scale=self.attention_scale,
            )
        else:
            total_len = offset + seq_len
            scores = (q @ k.transpose(-2, -1)) * self.attention_scale
            mask = self.causal_mask[offset:total_len, :total_len]
            if key_padding_mask is not None:
                mask = mask.unsqueeze(0) & key_padding_mask.to(torch.bool).unsqueeze(1)
                mask = mask | ~mask.any(dim=-1, keepdim=True)
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(~mask, float("-inf"))
            probs = self.attn_dropout(F.softmax(scores, dim=-1))
            out = probs @ v

        out = out.transpose(1, 2).contiguous().view(bsz, seq_len, dim)
        out = out * torch.sigmoid(self.gate_proj(x))
        out = self.resid_dropout(self.out_proj(out))

        if use_cache:
            return out, cached_kv
        return out

Writing attention.py


In [4]:
%%writefile model_arch.py
import math
from dataclasses import dataclass
from typing import Optional, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.utils import ModelOutput
from attention import CausalSelfAttention


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 560
    n_layers: int = 8
    n_heads: int = 8
    d_ff: int = 2240
    max_seq_len: int = 1024
    dropout: float = 0
    pos_emb_type: str = "rope"
    attention_scale: float = None
    norm_first: bool = True
    bos_token_id: int = None
    eos_token_id: int = None


@dataclass
class CausalLMOutput(ModelOutput):
    loss: Optional[torch.Tensor] = None
    logits: Optional[torch.Tensor] = None
    past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    scale = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model, dtype=torch.float32)
    pe[:, 0::2] = torch.sin(pos * scale)
    pe[:, 1::2] = torch.cos(pos * scale)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = getattr(cfg, 'norm_first', True)
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(
            d_model=cfg.d_model,
            n_heads=cfg.n_heads,
            max_seq_len=cfg.max_seq_len,
            dropout=cfg.dropout,
            attention_scale=cfg.attention_scale,
            use_rope=(cfg.pos_emb_type == "rope")
        )
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x: torch.Tensor, past_kv=None, use_cache: bool = False, key_padding_mask=None):
        if self.norm_first:
            attn_out = self.attn(self.ln1(x), past_kv=past_kv, use_cache=use_cache, key_padding_mask=key_padding_mask)
            if use_cache:
                attn_out, present = attn_out
            x = x + attn_out
            x = x + self.ffn(self.ln2(x))
        else:
            attn_out = self.attn(x, past_kv=past_kv, use_cache=use_cache, key_padding_mask=key_padding_mask)
            if use_cache:
                attn_out, present = attn_out
            x = self.ln1(x + attn_out)
            x = self.ln2(x + self.ffn(x))
        if use_cache:
            return x, present
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor = None, labels: torch.Tensor = None, idx: torch.Tensor = None,
                past_key_values: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None,
                use_cache: bool = False,
                attention_mask: Optional[torch.Tensor] = None,
                last_token_only: bool = False):
        if idx is None:
            idx = input_ids
        assert idx is not None, "Either input_ids or idx must be provided"

        bsz, seq_len = idx.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        assert past_len + seq_len <= self.cfg.max_seq_len,             f"Seq length {past_len + seq_len} exceeds max sequence length {self.cfg.max_seq_len}"

        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            if attention_mask is None:
                x = x + self.pos_emb[past_len:past_len + seq_len]
            else:
                pos_ids = (attention_mask.long().cumsum(dim=-1) - 1).clamp(min=0)[:, -seq_len:]
                x = x + self.pos_emb[pos_ids]

        presents = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past_kv = None if past_key_values is None else past_key_values[i]
            if use_cache:
                x, present = block(x, past_kv=past_kv, use_cache=True, key_padding_mask=attention_mask)
                presents.append(present)
            else:
                x = block(x, past_kv=past_kv, key_padding_mask=attention_mask)

        if last_token_only:
            x = x[:, -1:, :]
        x = self.ln_f(x)
        logits = self.head(x)

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous().view(-1, self.cfg.vocab_size)
            shift_labels = labels[:, 1:].contiguous().view(-1)
            loss = F.cross_entropy(shift_logits, shift_labels, ignore_index=-100)
            return CausalLMOutput(loss=loss, logits=logits, past_key_values=presents)

        if use_cache:
            return logits, presents
        return logits

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0, top_p: float = 1.0, eos_token_id: int = None, attention_mask: torch.Tensor = None,
                 stop_check=None, check_every: int = 16) -> torch.Tensor:
        was_training = self.training
        self.eval()

        bsz = idx.size(0)
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=idx.device, dtype=torch.bool)
        finished = torch.zeros(bsz, dtype=torch.bool, device=idx.device)

        prompt_len = idx.size(1)
        past_key_values = None
        cond_mask = None
        for step in range(max_new_tokens):
            if past_key_values is None or past_key_values[0][0].size(2) + 1 > self.cfg.max_seq_len:
                past_key_values = None
                idx_cond = idx[:, -self.cfg.max_seq_len:]
                cond_mask = None if attention_mask is None else attention_mask[:, -self.cfg.max_seq_len:]
            else:
                idx_cond = idx[:, -1:]

            logits, past_key_values = self(idx_cond, past_key_values=past_key_values, use_cache=True,
                                           attention_mask=cond_mask, last_token_only=True)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
                logits = logits.masked_fill(indices_to_remove, float('-inf'))

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            if eos_token_id is not None:
                next_token = torch.where(finished.unsqueeze(1), torch.full_like(next_token, eos_token_id), next_token)
                finished = finished | (next_token.squeeze(1) == eos_token_id)

            idx = torch.cat((idx, next_token), dim=1)
            if attention_mask is not None:
                ones = torch.ones(bsz, 1, dtype=torch.bool, device=idx.device)
                attention_mask = torch.cat((attention_mask, ones), dim=1)
                cond_mask = torch.cat((cond_mask, ones), dim=1)

            if eos_token_id is not None and finished.all():
                break

            if stop_check is not None and (step + 1) % check_every == 0:
                satisfied = torch.as_tensor(stop_check(idx[:, prompt_len:]), dtype=torch.bool, device=idx.device)
                if (satisfied | finished).all():
                    break

        if was_training:
            self.train()
        return idx

    @staticmethod
    def _repeated_ngram_tokens(seq, n):
        if len(seq) < n - 1:
            return []
        prefix = tuple(seq[len(seq) - (n - 1):])
        banned = []
        for i in range(len(seq) - n + 1):
            if tuple(seq[i:i + n - 1]) == prefix:
                banned.append(seq[i + n - 1])
        return banned

    @torch.no_grad()
    def generate_beam(self, idx: torch.Tensor, max_new_tokens: int, num_beams: int = 4, temperature: float = 1.0,
                      attention_mask: torch.Tensor = None, no_repeat_ngram_size: int = 0, banned_token_ids=None,
                      stop_check=None, check_every: int = 16) -> torch.Tensor:
        was_training = self.training
        self.eval()

        bsz, seq_len = idx.shape
        nb = num_beams
        device = idx.device
        max_new_tokens = min(max_new_tokens, self.cfg.max_seq_len - seq_len)

        if attention_mask is None:
            attention_mask = torch.ones(bsz, seq_len, dtype=torch.bool, device=device)
        attention_mask = attention_mask.to(device=device, dtype=torch.bool)

        logits, past = self(idx, use_cache=True, attention_mask=attention_mask, last_token_only=True)
        past = [(k.repeat_interleave(nb, dim=0), v.repeat_interleave(nb, dim=0)) for k, v in past]
        mask = attention_mask.repeat_interleave(nb, dim=0)
        next_logits = logits[:, -1, :].repeat_interleave(nb, dim=0)

        beam_scores = torch.zeros(bsz, nb, device=device)
        beam_scores[:, 1:] = float("-inf")
        gen = torch.empty(bsz * nb, 0, dtype=torch.long, device=device)
        row_offset = (torch.arange(bsz, device=device) * nb).unsqueeze(1)

        for step in range(max_new_tokens):
            if step > 0:
                mask = torch.cat([mask, torch.ones(bsz * nb, 1, dtype=torch.bool, device=device)], dim=1)
                out, past = self(gen[:, -1:], past_key_values=past, use_cache=True,
                                 attention_mask=mask, last_token_only=True)
                next_logits = out[:, -1, :]

            log_probs = F.log_softmax(next_logits.float() / max(temperature, 1e-5), dim=-1)

            if banned_token_ids:
                log_probs[:, banned_token_ids] = float("-inf")

            if no_repeat_ngram_size > 0 and gen.size(1) >= no_repeat_ngram_size - 1:
                for r, seq in enumerate(gen.tolist()):
                    bad = self._repeated_ngram_tokens(seq, no_repeat_ngram_size)
                    if bad:
                        log_probs[r, bad] = float("-inf")

            vocab = log_probs.size(-1)
            scores = (beam_scores.view(-1, 1) + log_probs).reshape(bsz, nb * vocab)
            beam_scores, top_idx = scores.topk(nb, dim=-1)

            src_beam = (top_idx // vocab + row_offset).view(-1)
            next_tok = (top_idx % vocab).view(-1, 1)

            gen = torch.cat([gen[src_beam], next_tok], dim=1)
            past = [(k[src_beam], v[src_beam]) for k, v in past]

            if stop_check is not None and (step + 1) % check_every == 0:
                best_so_far = gen.view(bsz, nb, -1)[:, 0]
                satisfied = torch.as_tensor(stop_check(best_so_far), dtype=torch.bool)
                if satisfied.all():
                    break

        best = gen.view(bsz, nb, -1)[:, 0]
        if was_training:
            self.train()
        return torch.cat([idx, best], dim=1)

Writing model_arch.py


In [6]:
%%writefile tokenizer_bpe.py
import os
import json
from transformers import PreTrainedTokenizer
from huggingface_hub import hf_hub_download


class MyCustomTokenizer(PreTrainedTokenizer):
    END_OF_WORD = "</w>"

    def __init__(self, vocab_file, merges_file, pad_token="<pad>", unk_token="<unk>",
                 bos_token="<s>", eos_token="</s>", **kwargs):
        with open(vocab_file, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        self.id_to_token = {i: t for t, i in self.vocab.items()}

        self.byte_to_char = self._func_byte_to_char()
        self.char_to_byte = {c: b for b, c in self.byte_to_char.items()}

        self.merge_ranks = self._load_merges(merges_file)
        self.word_cache = {}

        kwargs.setdefault("clean_up_tokenization_spaces", False)
        super().__init__(
            pad_token=pad_token,
            unk_token=unk_token,
            bos_token=bos_token,
            eos_token=eos_token,
            **kwargs
        )

    @staticmethod
    def _func_byte_to_char():
        printable = (list(range(ord("!"), ord("~") + 1))
                     + list(range(ord("¡"), ord("¬") + 1))
                     + list(range(ord("®"), ord("ÿ") + 1)))
        byte_values = printable[:]
        char_codes = printable[:]
        extra = 0
        for b in range(256):
            if b not in printable:
                byte_values.append(b)
                char_codes.append(256 + extra)
                extra += 1
        return {b: chr(c) for b, c in zip(byte_values, char_codes)}

    def _load_merges(self, merges_file):
        pairs = []
        with open(merges_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#version"):
                    continue
                parts = line.split(" ")
                if len(parts) == 2:
                    pairs.append((parts[0], parts[1]))

        is_byte_ids = len(pairs) > 0 and pairs[0][0].startswith("b:")

        merge_ranks = {}
        for left, right in pairs:
            if is_byte_ids:
                left = self._byte_ids_to_symbol(left)
                right = self._byte_ids_to_symbol(right)
            merge_ranks[(left, right)] = len(merge_ranks)
        return merge_ranks

    def _byte_ids_to_symbol(self, text):
        symbol = ""
        for part in text.split("+"):
            if part == self.END_OF_WORD:
                symbol += part
            else:
                symbol += self.byte_to_char[int(part[2:])]
        return symbol

    @property
    def vocab_size(self):
        return len(self.vocab)

    def get_vocab(self):
        return self.vocab

    def _split_word(self, word):
        if word in self.word_cache:
            return self.word_cache[word]

        symbols = [self.byte_to_char[b] for b in word.encode("utf-8")] + [self.END_OF_WORD]

        while len(symbols) > 1:
            pairs = [(symbols[i], symbols[i + 1]) for i in range(len(symbols) - 1)]
            known_pairs = [p for p in pairs if p in self.merge_ranks]
            if not known_pairs:
                break

            best_pair = min(known_pairs, key=lambda p: self.merge_ranks[p])

            merged_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best_pair:
                    merged_symbols.append(best_pair[0] + best_pair[1])
                    i += 2
                else:
                    merged_symbols.append(symbols[i])
                    i += 1
            symbols = merged_symbols

        tokens = [s if s in self.vocab else self.unk_token for s in symbols]

        if len(self.word_cache) > 500_000:
            self.word_cache.clear()
        self.word_cache[word] = tokens
        return tokens

    def _tokenize(self, text):
        tokens = []
        for word in text.split():
            tokens.extend(self._split_word(word))
        return tokens

    def convert_tokens_to_string(self, tokens):
        byte_list = []
        for token in tokens:
            if token in self.all_special_tokens:
                continue
            ends = token.endswith(self.END_OF_WORD)
            if ends:
                token = token[:-len(self.END_OF_WORD)]
            byte_list.extend(self.char_to_byte[c] for c in token if c in self.char_to_byte)
            if ends:
                byte_list.append(ord(" "))
        return bytes(byte_list).decode("utf-8", errors="replace").strip()

    def _convert_token_to_id(self, token):
        return self.vocab.get(token, self.vocab.get(self.unk_token, 1))

    def _convert_id_to_token(self, index):
        return self.id_to_token.get(index, self.unk_token)

    def save_vocabulary(self, save_directory, filename_prefix=None):
        prefix = (filename_prefix + "-") if filename_prefix else ""
        vocab_file = os.path.join(save_directory, prefix + "vocab.json")
        merges_file = os.path.join(save_directory, prefix + "merges.txt")
        with open(vocab_file, "w", encoding="utf-8") as f:
            json.dump(self.vocab, f, ensure_ascii=False, indent=2)
        with open(merges_file, "w", encoding="utf-8") as f:
            f.write("#version: 0.2\n")
            for (left, right), _ in sorted(self.merge_ranks.items(), key=lambda item: item[1]):
                f.write(f"{left} {right}\n")
        return (vocab_file, merges_file)

Writing tokenizer_bpe.py


In [7]:
from model_arch import DecoderLM, LMConfig
from tokenizer_bpe import MyCustomTokenizer

TOKENIZER_REPO = "Granth2/cs781-custom-bpe-30k-Version-10-40-40-10"
vocab_path = hf_hub_download(TOKENIZER_REPO, "vocab.json", token=HF_TOKEN)
merges_path = hf_hub_download(TOKENIZER_REPO, "merges.txt", token=HF_TOKEN)
tok = MyCustomTokenizer(vocab_file=vocab_path, merges_file=merges_path)
print("tokenizer: ", type(tok).__name__, ", vocabulary:", len(tok))

weights_path = hf_hub_download(PHASE1_REPO, "pytorch_model.bin", token=HF_TOKEN)

phase1_state = torch.load(weights_path, map_location="cpu")
phase1_state = {k.replace("module.", "", 1): v for k, v in phase1_state.items()}

VOCAB = phase1_state["tok_emb.weight"].shape[0]

assert VOCAB == len(tok), f"tokenizer has {len(tok)} tokens but the checkpoint has {VOCAB}"

lm_cfg = LMConfig(vocab_size=VOCAB, max_seq_len=Context_window, d_model=N_EMBD, n_layers=N_LAYER,
                  n_heads=N_HEAD, d_ff=4 * N_EMBD, dropout=0)

def load_base():
    m = DecoderLM(lm_cfg)
    m.load_state_dict(phase1_state, strict=True)
    return m

_m = load_base()
print(f"Phase 1 model: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M parameters")
del _m

vocab.json:   0%|          | 0.00/640k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/320k [00:00<?, ?B/s]

tokenizer:  MyCustomTokenizer , vocabulary: 30000


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  461MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Phase 1 model: 115.1M parameters


In [8]:
def get(name):
    return hf_hub_download(DATA_REPO, f"Phase_2_LNER/{name}", repo_type="dataset", token=HF_TOKEN)

labels = [line.strip() for line in open(get("Train/labels.txt"), encoding="utf-8") if line.strip()]
tags = ["O"] + [f"{prefix}-{label}" for label in labels for prefix in ("B", "I")]
tag2id = {tag: idx for idx, tag in enumerate(tags)}
O_ID = tag2id["O"]

texts, gold_raw = {}, {}
texts.update(json.load(open(get("Train/fold1-text.json"), encoding="utf-8")))
gold_raw.update(json.load(open(get("Train/fold1-gold.json"), encoding="utf-8")))
texts.update(json.load(open(get("Train/fold2-text.json"), encoding="utf-8")))
gold_raw.update(json.load(open(get("Train/fold2-gold.json"), encoding="utf-8")))

test_texts = json.load(open(get("Test/fold3-text.json"), encoding="utf-8"))

d0 = next(iter(gold_raw))
print("raw gold sample:", gold_raw[d0][:2])

def proper_data(ents):
    out = []
    for item in ents:
        if isinstance(item, dict):
            out.append((int(item["start"]), int(item["end"]), item["label"]))
        else:
            out.append((int(item[0]), int(item[1]), item[2]))
    return out

gold = {doc_id: proper_data(gold_raw.get(doc_id, [])) for doc_id in texts}
ids_all = sorted(texts)
train_ids, val_ids = train_test_split(ids_all, test_size=0.15, random_state=SEED)
print(len(train_ids), "train docs |", len(val_ids), "val docs |", len(test_texts), "test docs")
print(labels)

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

raw gold sample: [[137, 153, 'RESP', 'STATE OF HARYANA'], [252, 261, 'DATE', '19.8.2011']]
59 train docs | 11 val docs | 35 test docs
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [9]:
enc_cache, tag_cache = {}, {}

SPLIT_PUNCT = True
LEAD_PUNCT = "([{" + '"' + "'"
TRAIL_PUNCT = ",.;:)]}!?" + '"' + "'"

def split_punct(word):
    if not SPLIT_PUNCT:
        return [(0, word)]
    lead_idx = 0
    while lead_idx < len(word) and word[lead_idx] in LEAD_PUNCT:
        lead_idx += 1
    tail_idx, tail_tokens = len(word), []
    while tail_idx > lead_idx:
        if word[tail_idx - 1] in TRAIL_PUNCT:
            tail_idx -= 1
            tail_tokens.append((tail_idx, word[tail_idx]))
        elif tail_idx - 2 > lead_idx and word[tail_idx - 2:tail_idx] in ("'s", "\u2019s"):
            tail_idx -= 2
            tail_tokens.append((tail_idx, word[tail_idx:tail_idx + 2]))
        else:
            break
    if tail_idx == lead_idx:
        return [(0, word)]
    tokens = [(0, word[:lead_idx])] if lead_idx else []
    tokens.append((lead_idx, word[lead_idx:tail_idx]))
    return tokens + tail_tokens[::-1]

def spaced(text):
    return " ".join(sub for word in text.split() for _, sub in split_punct(word))

def _encode_word(tokenizer, word, start_offset):
    pieces = tokenizer._split_word(word)
    piece_sizes = []
    for piece in pieces:
        if piece == tokenizer.unk_token:
            piece_sizes.append(None)
        else:
            body = piece[:-len(tokenizer.END_OF_WORD)] if piece.endswith(tokenizer.END_OF_WORD) else piece
            piece_sizes.append(sum(1 for c in body if c in tokenizer.char_to_byte))
    total_bytes = len(word.encode("utf-8"))
    known_bytes = sum(s for s in piece_sizes if s is not None)
    unk_count = sum(s is None for s in piece_sizes)
    if unk_count:
        share = max(total_bytes - known_bytes, 0) // unk_count
        piece_sizes = [share if s is None else s for s in piece_sizes]
    byte2char = []
    for char_idx, char in enumerate(word):
        byte2char.extend([char_idx] * len(char.encode("utf-8")))
    token_ids, offsets, cur_byte = [], [], 0
    for piece, size in zip(pieces, piece_sizes):
        if size == 0:
            start_char = end_char = start_offset + len(word)
        else:
            lo = min(cur_byte, total_bytes - 1)
            hi = min(cur_byte + size, total_bytes) - 1
            start_char, end_char = start_offset + byte2char[lo], start_offset + byte2char[hi] + 1
        cur_byte += size
        token_ids.append(tokenizer._convert_token_to_id(piece))
        offsets.append((start_char, end_char))
    return token_ids, offsets

def encode_with_offsets(tokenizer, text):
    token_ids, offsets = [], []
    cursor = 0
    for word in text.split():
        base_idx = text.index(word, cursor)
        cursor = base_idx + len(word)
        for off, sub_piece in split_punct(word):
            sub_ids, sub_offsets = _encode_word(tokenizer, sub_piece, base_idx + off)
            token_ids += sub_ids
            offsets += sub_offsets
    return token_ids, offsets

def get_enc(key, text):
    if key not in enc_cache:
        enc_cache[key] = encode_with_offsets(tok, text)
    return enc_cache[key]

def tags_for(offsets, entities):
    ends = [b for _, b in offsets]
    tag_list = [-100 if a == b else O_ID for a, b in offsets]
    for ent_start, ent_end, ent_label in entities:
        idx = bisect_right(ends, ent_start)
        is_first = True
        while idx < len(offsets):
            start_c, end_c = offsets[idx]
            if start_c == end_c:
                if start_c > ent_end:
                    break
                idx += 1
                continue
            if start_c >= ent_end:
                break
            prefix = "B-" if is_first else "I-"
            tag_list[idx] = tag2id[prefix + ent_label]
            is_first = False
            idx += 1
    return tag_list

def decode(tag_ids, offsets, text):
    spans, cur = [], None
    def flush_span():
        nonlocal cur
        if cur:
            s, e, lab = cur
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            if e > s:
                spans.append((s, e, lab))
        cur = None
    for t_id, (start_c, end_c) in zip(tag_ids, offsets):
        if start_c == end_c:
            continue
        tag_name = tags[t_id]
        if tag_name == "O":
            flush_span()
            continue
        prefix, label_str = tag_name.split("-", 1)
        if prefix == "B" or cur is None or cur[2] != label_str:
            flush_span()
            cur = [start_c, end_c, label_str]
        else:
            cur[1] = end_c
    flush_span()
    return spans

def window_starts(n):
    step = MAX_LEN - STRIDE
    starts = list(range(0, max(n, 1), step))
    while len(starts) > 1 and starts[-2] + MAX_LEN >= n:
        starts.pop()
    return starts

def window_plan(n):
    starts, half, plan = window_starts(n), STRIDE // 2, []
    for w_idx, start_pos in enumerate(starts):
        win_len = min(MAX_LEN, n - start_pos)
        lo = start_pos + (half if w_idx > 0 else 0)
        hi = start_pos + win_len - (half if w_idx < len(starts) - 1 else 0)
        plan.append((start_pos, lo, hi))
    return plan

def make_train_windows(doc_ids, keep_empty_prob=1.0):
    windows = []
    for doc_id in doc_ids:
        ids, offsets = get_enc(doc_id, texts[doc_id])
        if doc_id not in tag_cache:
            tag_cache[doc_id] = tags_for(offsets, gold[doc_id])
        doc_tags = tag_cache[doc_id]
        for start_pos in window_starts(len(ids)):
            win_tags = doc_tags[start_pos:start_pos + MAX_LEN]
            if not any(t > O_ID for t in win_tags) and random.random() > keep_empty_prob:
                continue
            windows.append(dict(ids=ids[start_pos:start_pos + MAX_LEN], tags=win_tags))
    return windows

def strict_scores(predictions, targets):
    tp, fp, fn = defaultdict(int), defaultdict(int), defaultdict(int)
    for doc_id in targets:
        p_set = {tuple(x[:3]) for x in predictions.get(doc_id, [])}
        g_set = {tuple(x[:3]) for x in targets[doc_id]}
        for item in p_set & g_set:
            tp[item[2]] += 1
        for item in p_set - g_set:
            fp[item[2]] += 1
        for item in g_set - p_set:
            fn[item[2]] += 1
    f1_scores = {}
    for label_name in labels:
        p = tp[label_name] / (tp[label_name] + fp[label_name]) if tp[label_name] + fp[label_name] else 0
        r = tp[label_name] / (tp[label_name] + fn[label_name]) if tp[label_name] + fn[label_name] else 0
        f1_scores[label_name] = 2 * p * r / (p + r) if p + r else 0
    return sum(f1_scores.values()) / len(f1_scores), f1_scores

In [10]:
for d in train_ids[:5]:
    ids, offs = get_enc(d, texts[d])
    assert ids == tok.encode(spaced(texts[d]), add_special_tokens=False), f"token mismatch in doc {d}"
print("token ids match tok.encode")

def oracle(doc_ids):
    pred = {}
    for d in doc_ids:
        ids, offs = get_enc(d, texts[d])
        if d not in tag_cache:
            tag_cache[d] = tags_for(offs, gold[d])
        pred[d] = decode(tag_cache[d], offs, texts[d])
    return strict_scores(pred, {d: gold[d] for d in doc_ids})

def show_misses(doc_ids, n=10):
    shown = 0
    for d in doc_ids:
        ids, offs = get_enc(d, texts[d])
        got = set(decode(tag_cache[d], offs, texts[d]))
        for s, e, l in gold[d]:
            if (s, e, l) not in got:
                t = texts[d]
                print(l, repr(t[max(0, s - 15):s]), "[[", repr(t[s:e]), "]]", repr(t[e:e + 15]))
                shown += 1
                if shown >= n:
                    return

macro, per = oracle(val_ids)
print("oracle strict macro-F1 on val:", round(macro, 4))
print({k: round(v, 3) for k, v in per.items()})
show_misses(val_ids)

token ids match tok.encode
oracle strict macro-F1 on val: 0.9837
{'APP': 0.979, 'RESP': 1.0, 'A.COUNSEL': 0.973, 'R.COUNSEL': 0.98, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.982, 'COURT': 0.963, 'STAT': 0.96, 'PREC': 0.972, 'DATE': 1.0, 'CASENO': 0.995}
STAT 'C (3) 183 ACT: ' [[ 'Orissa Estates Abolition Act, 1951' ]] '-Secs. 26-27- C'
AUTH 'econdly by the ' [[ 'Chief Conservator of Forests' ]] ',screening it a'
APP 'NTS: - 1. Smt. ' [[ 'Mangala ' ]] 'wd/o Vijay Khan'
PREC 'ry, before the ' [[ 'Consumer Forum' ]] 's there were tw'
STAT '......T..J The ' [[ 'Constitution' ]] 'al validity of '
STAT 'To support the ' [[ 'Constitution' ]] 'al validity of '
COURT 'nd some of the ' [[ 'High Court' ]] 's in India in r'
PREC ' reiterated in ' [[ 'State of Haryana v. Mohinder Singh [2000 (3) SCC 394]' ]] '[email protecte'
COURT 'dgments of the ' [[ 'High Court' ]] 's, this Court c'
COURT 'tleast in many ' [[ 'High Court' ]] 's. As the prese'


In [11]:
from peft import LoraConfig, inject_adapter_in_model

def backbone_hidden(model, input_ids):
    assert model.cfg.pos_emb_type == "rope", "add the sinusoidal branch if you switch position type"
    hidden = model.tok_emb(input_ids)
    for block in model.blocks:
        hidden = block(hidden)
    return model.ln_f(hidden)

class NERModel(nn.Module):
    def __init__(self, backbone, hidden, n_tags, dropout=0.1, rnn_hidden=256):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.rnn = nn.GRU(hidden, rnn_hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(2 * rnn_hidden, n_tags)

    def forward(self, input_ids):
        h = backbone_hidden(self.backbone, input_ids)
        seq_len = input_ids.size(1)
        pos = torch.arange(1, seq_len + 1, device=input_ids.device)[None]
        lengths = ((input_ids != PAD) * pos).max(1).values.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(self.drop(h).float(), lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=seq_len)
        return self.head(self.drop(out))

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def adapt_lora(base, r=8, alpha=16, dropout=0.05, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in base.parameters():
        p.requires_grad = False
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=list(targets), bias="none")
    base = inject_adapter_in_model(cfg, base)
    for n, p in base.named_parameters():
        p.requires_grad = "lora_" in n
    return base

def adapt_full(base):
    for p in base.parameters():
        p.requires_grad = True
    return base

def adapt_top_layers(base, n_layers=6):
    for p in base.parameters():
        p.requires_grad = False
    for block in base.blocks[-n_layers:]:
        for p in block.parameters():
            p.requires_grad = True
    for p in base.ln_f.parameters():
        p.requires_grad = True
    return base

class LoRMALinear(nn.Module):
    def __init__(self, base_linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base_linear
        self.base.weight.requires_grad = False
        d_out = base_linear.out_features
        self.scale = alpha / r
        self.lorma_A = nn.Parameter(torch.empty(r, d_out))
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.lorma_A, a=5 ** 0.5)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        y0 = self.base(x)
        delta = (self.drop(y0) @ self.lorma_A.t()) @ self.lorma_B.t()
        return y0 + self.scale * delta

def adapt_lorma(base, r=8, alpha=16, dropout=0.05, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in base.parameters():
        p.requires_grad = False
    for module in list(base.modules()):
        for name, child in list(module.named_children()):
            if name in targets and isinstance(child, nn.Linear):
                setattr(module, name, LoRMALinear(child, r, alpha, dropout))
    for n, p in base.named_parameters():
        p.requires_grad = "lorma_" in n
    return base

In [12]:
from transformers.optimization import get_cosine_schedule_with_warmup

PAD = tok.pad_token_id if tok.pad_token_id is not None else 0
use_amp = device == "cuda"

def collate(batch):
    max_len = max(len(b["ids"]) for b in batch)
    x = torch.full((len(batch), max_len), PAD, dtype=torch.long)
    y = torch.full((len(batch), max_len), -100, dtype=torch.long)
    for i, item in enumerate(batch):
        length = len(item["ids"])
        x[i, :length] = torch.tensor(item["ids"])
        if item.get("tags") is not None:
            y[i, :length] = torch.tensor(item["tags"])
    return x, y

def tag_class_weights(windows, power=0.5):
    counts = np.ones(len(tags))
    for win in windows:
        for t in win["tags"]:
            if t >= 0:
                counts[t] += 1
    weights = (counts.sum() / counts) ** power
    return torch.tensor(weights / weights.mean(), dtype=torch.float)

O_BIAS = 0.0

@torch.no_grad()
def predict(model, doc_texts, prefix="", bs=8):
    model.to(device).eval()
    result = {}
    for doc_id, text in doc_texts.items():
        ids, offsets = get_enc(prefix + doc_id, text)
        n = len(ids)
        if n == 0:
            result[doc_id] = []
            continue
        plan = window_plan(n)
        pred_tags = [O_ID] * n
        for i in range(0, len(plan), bs):
            group = plan[i:i + bs]
            x, _ = collate([dict(ids=ids[st:st + MAX_LEN]) for st, _, _ in group])
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x.to(device))
            logits = logits.float()
            logits[..., O_ID] -= O_BIAS
            preds = logits.argmax(-1).cpu()
            for k, (st, lo, hi) in enumerate(group):
                if hi > lo:
                    pred_tags[lo:hi] = preds[k, lo - st:hi - st].tolist()
        result[doc_id] = [list(span) for span in decode(pred_tags, offsets, text)]
    return result

def evaluate(model, doc_ids):
    predictions = predict(model, {d: texts[d] for d in doc_ids})
    return strict_scores(predictions, {d: gold[d] for d in doc_ids})

def train(model, windows, epochs, lr, bs, weights=None, val=None, head_lr=None):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    if head_lr is None:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    else:
        backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
        head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
        opt = torch.optim.AdamW([{"params": backbone_params, "lr": lr},
                                 {"params": head_params, "lr": head_lr}], weight_decay=0.01)
    steps = epochs * ((len(windows) + bs - 1) // bs)
    sched = get_cosine_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    criterion = nn.CrossEntropyLoss(weight=None if weights is None else weights.to(device), ignore_index=-100)
    best_f1, best_state = -1.0, None
    for ep in range(epochs):
        model.train()
        perm = np.random.permutation(len(windows))
        total_loss, seen = 0.0, 0
        for i in range(0, len(perm), bs):
            x, y = collate([windows[j] for j in perm[i:i + bs]])
            x, y = x.to(device), y.to(device)
            if (y != -100).sum() == 0:
                continue
            with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x)
            loss = criterion(logits.float().view(-1, len(tags)), y.view(-1))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            total_loss += loss.item()
            seen += 1
        msg = f"epoch {ep + 1}/{epochs} | train loss {total_loss / max(seen, 1):.4f}"
        if val is not None:
            f1, _ = evaluate(model, val)
            msg += f" | val strict macro-F1 {f1:.4f}"
            if f1 > best_f1:
                best_f1 = f1
                best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        print(msg)
    if best_state is not None:
        with torch.no_grad():
            for n, p in model.named_parameters():
                if n in best_state:
                    p.copy_(best_state[n].to(p.device))
    return best_f1

def save_trainable(model, path):
    torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, path)

In [13]:
results = []
train_windows = make_train_windows(train_ids + val_ids, KEEP_EMPTY_PROB)
class_w = tag_class_weights(train_windows)
print(len(train_windows), "training windows")

def run_experiment(name, adapt_fn, epochs=EPOCHS, lr=LR, bs=BATCH, use_weights=True, head_lr=None):
    base_model = adapt_fn(load_base())
    model = NERModel(base_model, N_EMBD, len(tags))
    n_trainable, n_all = count_trainable(model)
    print(f"[{name}] trainable {n_trainable/1e6:.2f}M of {n_all/1e6:.1f}M")
    start_time = time.time()
    best = train(model, train_windows, epochs, lr, bs, class_w if use_weights else None, None, head_lr)
    mins = (time.time() - start_time) / 60
    results.append(dict(name=name, trainable=n_trainable, total=n_all, minutes=mins))
    save_trainable(model, f"{name}.pt")
    print(f"[{name}] trained on {len(train_ids) + len(val_ids)} docs (train + val) | {mins:.1f} min")
    return model

1196 training windows


In [14]:
FINETUNE = "full"
BACKBONE_LR = 5e-5
NEW_LAYER_LR = 5e-4
FINAL_EPOCHS = 15

adapt_fn = adapt_full if FINETUNE == "full" else adapt_top_layers
model_ft = run_experiment(f"{FINETUNE}_finetune_bigru", adapt_fn,
                          epochs=FINAL_EPOCHS, lr=BACKBONE_LR, head_lr=NEW_LAYER_LR)

[full_finetune_bigru] trainable 116.73M of 116.7M


/tmp/ipykernel_23/455719450.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


epoch 1/15 | train loss 1.1731
epoch 2/15 | train loss 0.1840
epoch 3/15 | train loss 0.0739
epoch 4/15 | train loss 0.0378
epoch 5/15 | train loss 0.0215
epoch 6/15 | train loss 0.0108
epoch 7/15 | train loss 0.0067
epoch 8/15 | train loss 0.0040
epoch 9/15 | train loss 0.0025
epoch 10/15 | train loss 0.0019
epoch 11/15 | train loss 0.0014
epoch 12/15 | train loss 0.0012
epoch 13/15 | train loss 0.0011
epoch 14/15 | train loss 0.0010
epoch 15/15 | train loss 0.0011
[full_finetune_bigru] trained on 70 docs (train + val) | 11.1 min


In [15]:
OPEN_PUNCT = "([{" + chr(34) + chr(39)

def snap_start_to_word(spans, text):
    snapped = []
    for start, end, label in spans:
        while start > 0 and not text[start - 1].isspace() and text[start - 1] not in OPEN_PUNCT:
            start -= 1
        snapped.append([start, end, label])
    return snapped

def write_submission(model, out_zip="submission.zip", snap=True):
    predictions = predict(model, test_texts, prefix="test:")
    if snap:
        predictions = {d: snap_start_to_word(spans, test_texts[d]) for d, spans in predictions.items()}
    for d in test_texts:
        predictions.setdefault(d, [])
    assert set(predictions) == set(test_texts)
    with open("prediction.json", "w", encoding="utf-8") as f:
        json.dump(predictions, f)
    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write("prediction.json", arcname="prediction.json")
    print("wrote", out_zip, "| docs:", len(predictions), "| spans:", sum(len(v) for v in predictions.values()))

write_submission(model_ft, "submission.zip")

O_BIAS = 1.0
write_submission(model_ft, "submission_obias1.zip")
O_BIAS = 0.0

wrote submission.zip | docs: 35 | spans: 4847
wrote submission_obias1.zip | docs: 35 | spans: 5000
